# Tarea 9 · Capa Gold

La publicación cuenta como día 1. El tercer día corresponde a `datediff = 2`. Las comparaciones usan únicamente ese día; no se sustituyen fechas ausentes ni se suman snapshots acumulados de varios días. La tasa agregada es el cociente de sumas de vídeos con reproducciones positivas.

In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'config' / 'gold.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/gold.json')
sys.path.insert(0, str(project_root / 'scripts'))
from gold_metrics import generar_gold
config = json.loads((project_root / 'config' / 'gold.json').read_text(encoding='utf-8'))
silver = json.loads((project_root / config['silver_config']).read_text(encoding='utf-8'))


## Lectura y transformación de Silver

Solo se leen publicaciones y mediciones vigentes. Se comprueba la unicidad y la correspondencia entre ambas tablas.

In [2]:
publicaciones = spark.table(silver['tables']['publicaciones'])
mediciones = spark.table(silver['tables']['mediciones_vigentes'])
gold = generar_gold(publicaciones, mediciones, config['dia_objetivo'])
print('Publicaciones:', publicaciones.count())
print('Mediciones vigentes:', mediciones.count())
assert gold['rendimiento_video'].count() == mediciones.count()
for nombre, df in gold.items():
    print(nombre, df.count())

Publicaciones: 40
Mediciones vigentes: 467
rendimiento_video 467
rendimiento_categoria 8
rendimiento_plataforma 2


## Escritura de tablas Delta

Gold se reconstruye a partir del estado vigente de Silver. Solo se reemplazan las tres tablas Gold del proyecto; Bronze, Silver y los checkpoints no se modifican

In [3]:
nombres = {'rendimiento_video', 'rendimiento_categoria', 'rendimiento_plataforma'}
assert config['schema'] == 'workspace.lensdata_gold'
assert set(config['tables']) == nombres
assert all(config['tables'][n] == f"{config['schema']}.{n}" for n in nombres)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['schema']}")
for nombre, df in gold.items():
    tabla = config['tables'][nombre]
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(tabla)
    print(f'{tabla}: {spark.table(tabla).count()} filas')

workspace.lensdata_gold.rendimiento_video: 467 filas
workspace.lensdata_gold.rendimiento_categoria: 8 filas
workspace.lensdata_gold.rendimiento_plataforma: 2 filas


## Resultados para el dashboard

In [4]:
video = spark.table(config['tables']['rendimiento_video'])
video.filter((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000')).orderBy('fecha_medicion').show(20, truncate=False)
spark.table(config['tables']['rendimiento_categoria']).orderBy('plataforma', F.col('tasa_interaccion').desc_nulls_last()).show(20, truncate=False)
spark.table(config['tables']['rendimiento_plataforma']).orderBy('plataforma').show(truncate=False)

+----------+--------+---------+-----------------+-----------------+--------------+---------+--------------+--------------+--------+-----------+-----------+-------------+-----------------+
|plataforma|video_id|categoria|titulo           |fecha_publicacion|fecha_medicion|dia_video|version_origen|reproducciones|me_gusta|comentarios|compartidos|interacciones|tasa_interaccion |
+----------+--------+---------+-----------------+-----------------+--------------+---------+--------------+--------------+--------+-----------+-----------+-------------+-----------------+
|tiktok    |V000    |regalos  |Vídeo de tienda 1|2026-11-01       |2026-11-01    |1        |2             |950           |47      |13         |10         |70           |7.368421052631579|
|tiktok    |V000    |regalos  |Vídeo de tienda 1|2026-11-01       |2026-11-02    |2        |1             |1200          |60      |17         |13         |90           |7.5              |
+----------+--------+---------+-----------------+-----------